# Produisez une étude de marché avec R ou Python


## Contexte

Je travaille chez La poule qui chante, une entreprise française qui élève et vend des poulets bio.
Pour le moment, l’entreprise vend seulement en France.

Le PDG souhaite savoir si l’entreprise pourrait vendre ses poulets dans d’autres pays.
Aucun pays n’est encore choisi. Tous les pays sont possibles.

**Objectif**

Mon travail est de :
*   Analyser différents pays du monde
*   Regrouper les pays qui se ressemblent
*   Identifier quels groupes de pays pourraient être intéressants pour exporter nos poulets

In [ ]:
#pestel
import pandas as pd
data = {
    "Dimension PESTEL": [
        "Politique", "Politique", "Économique", "Économique",
         "Socioculturel", "Socioculturel",
        "Technologique", "Environnemental", "Légal", "Légal"
    ],
    "Variable": [
        "Stabilité politique actuelle", "Stabilité politique sur 5 ans",
        "PIB par habitant", "Importation volailles / habitants",
        "Croissance sur 5 ans", "Disponibilité Volailles / habitants",
        "% de population ayant de l'électricité", "Distance depuis la france",
        "Taxes", "Accord-EU"
    ],
    "Objectif stratégique": [
        "Mesurer le risque pays", "Analyser la dynamique politique",
        "Évaluer le pouvoir d'achat", "Identifier les opportunités d'export",
        "Potentiel futur (Démographie)",
        "Habitudes alimentaires", "Fiabilité chaîne du froid",
        "Coût logistique", "Barrières douanières", "Facilité d’accès"
    ]
}
pestel_df = pd.DataFrame(data)
pestel_df

,Dimension PESTEL,Variable,Objectif stratégique
0,Politique,Stabilité politique actuelle,Mesurer le risque pays
1,Politique,Stabilité politique sur 5 ans,Analyser la dynamique politique
2,Économique,PIB par habitant,Évaluer le pouvoir d'achat
3,Économique,Importation volailles / habitants,Identifier les opportunités d'export
4,Socioculturel,Croissance sur 5 ans,Potentiel futur (Démographie)
5,Socioculturel,Disponibilité Volailles / habitants,Habitudes alimentaires
6,Technologique,% de population ayant de l'électricité,Fiabilité chaîne du froid
7,Environnemental,Distance depuis la france,Coût logistique
8,Légal,Taxes,Barrières douanières
9,Légal,Accord-EU,Facilité d’accès


## Importation des données

In [ ]:
#importation des fichiers csv depuis le drive
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

df_disponibilite_alimentaire = pd.read_csv('/content/drive/MyDrive/OC/DisponibiliteAlimentaire_2017.csv',sep=',')
df_population = pd.read_csv('/content/drive/MyDrive/OC/Population_2000_2018.csv',sep=',')
df_stabilites_politiques = pd.read_csv('/content/drive/MyDrive/OC/Stabilite_Politique_2017.csv',sep=',')
df_stabilites_politiques_evolution = pd.read_csv('/content/drive/MyDrive/OC/Stabilite_Politique_Evolution.csv',sep=',')
df_pib = pd.read_csv('/content/drive/MyDrive/OC/PIB_2017.csv',sep=',')
df_taxes = pd.read_csv('/content/drive/MyDrive/OC/taxes.csv',sep=';')
df_distance = pd.read_csv('/content/drive/MyDrive/OC/distances_from_france.csv',sep=',')
df_electricite = pd.read_csv('/content/drive/MyDrive/OC/electricity_2017.csv',sep=',')
df_accord_eu = pd.read_csv('/content/drive/MyDrive/OC/All_Countries_EU_Trade_Agreements_ISO3.csv',sep=',')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Standardisation


In [ ]:
fao_to_iso_map = {
    1: "ARM", 2: "AFG", 3: "ALB", 4: "DZA", 5: "ASM", 6: "AND", 7: "AGO", 8: "ATG",
    9: "ARG", 10: "AUS", 11: "AUT", 12: "BHS", 13: "BHR", 14: "BRB", 16: "BGD",
    17: "BMU", 18: "BTN", 19: "BOL", 20: "BWA", 21: "BRA", 22: "ABW", 23: "BLZ",
    25: "SLB", 26: "BRN", 27: "BGR", 28: "MMR", 29: "BDI", 32: "CMR", 33: "CAN",
    35: "CPV", 36: "CYM", 37: "CAF", 38: "LKA", 39: "TCD", 40: "CHL", 41: "CHN",
    44: "COL", 45: "COM", 46: "COG", 47: "COK", 48: "CRI", 49: "CUB", 50: "CYP",
    52: "AZE", 53: "BEN", 54: "DNK", 55: "DMA", 56: "DOM", 57: "BLR", 58: "ECU",
    59: "EGY", 60: "SLV", 61: "GNQ", 63: "EST", 64: "FRO", 65: "FLK", 66: "FJI",
    67: "FIN", 68: "FRA", 69: "GUF", 70: "PYF", 72: "DJI", 73: "GEO", 74: "GAB",
    75: "GMB", 79: "DEU", 80: "BIH", 81: "GHA", 82: "GIB", 83: "KIR", 84: "GRC",
    85: "GRL", 86: "GRD", 87: "GLP", 88: "GUM", 89: "GTM", 90: "GIN", 91: "GUY",
    93: "HTI", 94: "VAT", 95: "HND", 96: "HKG", 97: "HUN", 98: "HRV", 99: "ISL",
    100: "IND", 101: "IDN", 102: "IRN", 103: "IRQ", 104: "IRL", 105: "ISR", 106: "ITA",
    107: "CIV", 108: "KAZ", 109: "JAM", 110: "JPN", 112: "JOR", 113: "KGZ", 114: "KEN",
    115: "KHM", 116: "PRK", 117: "KOR", 118: "KWT", 119: "LVA", 120: "LAO", 121: "LBN",
    122: "LSO", 123: "LBR", 124: "LBY", 125: "LIE", 126: "LTU", 127: "MHL", 128: "MAC",
    129: "MDG", 130: "MWI", 131: "MYS", 132: "MDV", 133: "MLI", 134: "MLT", 135: "MTQ",
    136: "MRT", 137: "MUS", 138: "MEX", 140: "MCO", 141: "MNG", 142: "MSR", 143: "MAR",
    144: "MOZ", 145: "FSM", 146: "MDA", 147: "NAM", 148: "NRU", 149: "NPL", 150: "NLD",
    153: "NCL", 154: "MKD", 155: "VUT", 156: "NZL", 157: "NIC", 158: "NER", 159: "NGA",
    160: "NIU", 161: "NFK", 162: "NOR", 163: "MNP", 165: "PAK", 166: "PAN", 167: "CZE",
    168: "PNG", 169: "PRY", 170: "PER", 171: "PHL", 172: "PCN", 173: "POL", 174: "PRT",
    175: "GNB", 176: "TLS", 177: "PRI", 178: "ERI", 179: "QAT", 180: "PLW", 181: "ZWE",
    182: "REU", 183: "ROU", 184: "RWA", 185: "RUS", 188: "KNA", 189: "LCA", 190: "SPM",
    191: "VCT", 192: "SMR", 193: "STP", 194: "SAU", 195: "SEN", 196: "SYC", 197: "SLE",
    198: "SVN", 199: "SVK", 200: "SGP", 201: "SOM", 202: "ZAF", 203: "ESP", 205: "ESH",
    207: "SUR", 208: "TJK", 209: "SWZ", 210: "SWE", 211: "CHE", 212: "SYR", 213: "TKM",
    214: "TWN", 215: "TZA", 216: "THA", 217: "TGO", 218: "TKL", 219: "TON", 220: "TTO",
    221: "OMN", 222: "TUN", 223: "TUR", 224: "TCA", 225: "ARE", 226: "UGA", 227: "TUV",
    229: "GBR", 230: "UKR", 231: "USA", 233: "BFA", 234: "URY", 235: "UZB", 236: "VEN",
    237: "VNM", 238: "ETH", 239: "VGB", 240: "VIR", 243: "WLF", 244: "WSM", 249: "YEM",
    250: "COD", 251: "ZMB", 255: "BEL", 256: "LUX", 258: "AIA", 260: "SJM", 264: "IMN",
    270: "MYT", 272: "SRB", 273: "MNE", 276: "SDN", 277: "SSD", 299: "PSE"
}

## Lecture et nettoyage des données

### Disponibilité & Importation alimentaire

Data frame pour la disponibilité de volailles par pays

In [ ]:
proteines = ['Viande de Volailles']

In [ ]:
df_disponibilite_interieur_proteines = df_disponibilite_alimentaire[df_disponibilite_alimentaire.Produit.isin(proteines)]

In [ ]:
df_disponibilite_interieur_proteines = df_disponibilite_interieur_proteines.loc[df_disponibilite_interieur_proteines['Élément']=='Disponibilité intérieure']

In [ ]:
df_disponibilite_interieur_proteines.head()

,Code Domaine,Domaine,Code zone,Zone,Code Élément,Élément,Code Produit,Produit,Code année,Année,Unité,Valeur,Symbole,Description du Symbole
654,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5301,Disponibilité intérieure,2734,Viande de Volailles,2017,2017,Milliers de tonnes,57.0,S,Données standardisées
1708,FBS,Nouveaux Bilans Alimentaire,202,Afrique du Sud,5301,Disponibilité intérieure,2734,Viande de Volailles,2017,2017,Milliers de tonnes,2118.0,S,Données standardisées
2717,FBS,Nouveaux Bilans Alimentaire,3,Albanie,5301,Disponibilité intérieure,2734,Viande de Volailles,2017,2017,Milliers de tonnes,47.0,S,Données standardisées
3776,FBS,Nouveaux Bilans Alimentaire,4,Algérie,5301,Disponibilité intérieure,2734,Viande de Volailles,2017,2017,Milliers de tonnes,277.0,S,Données standardisées
4877,FBS,Nouveaux Bilans Alimentaire,79,Allemagne,5301,Disponibilité intérieure,2734,Viande de Volailles,2017,2017,Milliers de tonnes,1739.0,S,Données standardisées


In [ ]:
df_disponibilite_interieur_proteines = df_disponibilite_interieur_proteines.groupby(['Code zone','Zone'])['Valeur'].sum()
df_disponibilite_interieur_proteines = df_disponibilite_interieur_proteines.reset_index()
df_disponibilite_interieur_proteines = df_disponibilite_interieur_proteines.sort_values(by='Valeur',ascending=False)
df_disponibilite_interieur_proteines = df_disponibilite_interieur_proteines.rename(columns={'Valeur':'Disponibilité Volailles (Milliers de t)'})
df_disponibilite_interieur_proteines

,Code zone,Zone,Disponibilité Volailles (Milliers de t)
156,231,États-Unis d'Amérique,18266.0
26,41,"Chine, continentale",18161.0
14,21,Brésil,9982.0
126,185,Fédération de Russie,4556.0
97,138,Mexique,4219.0
...,...,...,...
16,25,Îles Salomon,3.0
107,155,Vanuatu,3.0
46,72,Djibouti,3.0
53,83,Kiribati,2.0


In [ ]:
df_disponibilite_interieur_proteines.info()

<class 'pandas.core.frame.DataFrame'>
Index: 170 entries, 156 to 130
Data columns (total 3 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Code zone                                170 non-null    int64  
 1   Zone                                     170 non-null    object 
 2   Disponibilité Volailles (Milliers de t)  170 non-null    float64
dtypes: float64(1), int64(1), object(1)
memory usage: 5.3+ KB


In [ ]:
#importation du code iso3
df_disponibilite_interieur_proteines_iso3 = pd.merge(df_disponibilite_interieur_proteines, pd.DataFrame(list(fao_to_iso_map.items()), columns=['Code zone', 'Code ISO3']), on='Code zone', how='left')

In [ ]:
df_disponibilite_interieur_proteines_iso3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 170 entries, 0 to 169
Data columns (total 4 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Code zone                                170 non-null    int64  
 1   Zone                                     170 non-null    object 
 2   Disponibilité Volailles (Milliers de t)  170 non-null    float64
 3   Code ISO3                                170 non-null    object 
dtypes: float64(1), int64(1), object(2)
memory usage: 5.4+ KB


In [122]:
#df_disponibilite_interieur_proteines_iso3[df_disponibilite_interieur_proteines_iso3['Code ISO3'] == 'KNA']

,Code zone,Zone,Disponibilité Volailles (Milliers de t),Code ISO3
164,188,Saint-Kitts-et-Nevis,4.0,KNA


In [ ]:
#affichage iso3 nul
#df_disponibilite_interieur_proteines_iso3.loc[df_disponibilite_interieur_proteines_iso3['Code ISO3'].isna()]

Data frame pour l'importation de volailles par pays


In [ ]:
volailles = ['Viande de Volailles']

In [ ]:
df_importation_volailles = df_disponibilite_alimentaire[df_disponibilite_alimentaire.Produit.isin(volailles)]

In [ ]:
df_importation_volailles = df_importation_volailles.loc[df_importation_volailles['Élément']=='Importations - Quantité']

In [ ]:
df_importation_volailles = df_importation_volailles.groupby(['Code zone','Zone'])['Valeur'].sum()
df_importation_volailles = df_importation_volailles.reset_index()
df_importation_volailles = df_importation_volailles.sort_values(by='Valeur',ascending=False)
df_importation_volailles = df_importation_volailles.rename(columns={'Valeur':'Importation volailles (Milliers de t)'})
df_importation_volailles

,Code zone,Zone,Importation volailles (Milliers de t)
75,110,Japon,1069.0
97,138,Mexique,972.0
61,96,Chine - RAS de Hong-Kong,907.0
50,79,Allemagne,842.0
154,229,Royaume-Uni de Grande-Bretagne et d'Irlande du...,779.0
...,...,...,...
125,184,Rwanda,0.0
132,195,Sénégal,0.0
150,222,Tunisie,0.0
153,226,Ouganda,0.0


In [ ]:
df_importation_volailles.info()

<class 'pandas.core.frame.DataFrame'>
Index: 170 entries, 75 to 157
Data columns (total 3 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Code zone                              170 non-null    int64  
 1   Zone                                   170 non-null    object 
 2   Importation volailles (Milliers de t)  170 non-null    float64
dtypes: float64(1), int64(1), object(1)
memory usage: 5.3+ KB


In [ ]:
#importation iso3
df_importation_volailles_iso3 = pd.merge(df_importation_volailles, pd.DataFrame(list(fao_to_iso_map.items()), columns=['Code zone', 'Code ISO3']), on='Code zone', how='left')

In [ ]:
df_importation_volailles_iso3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 170 entries, 0 to 169
Data columns (total 4 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Code zone                              170 non-null    int64  
 1   Zone                                   170 non-null    object 
 2   Importation volailles (Milliers de t)  170 non-null    float64
 3   Code ISO3                              170 non-null    object 
dtypes: float64(1), int64(1), object(2)
memory usage: 5.4+ KB


In [123]:
#df_importation_volailles_iso3[df_importation_volailles_iso3['Code ISO3'] == 'KNA']

,Code zone,Zone,Importation volailles (Milliers de t),Code ISO3
121,188,Saint-Kitts-et-Nevis,4.0,KNA


### Population & Croissance & PIB

Data frame nbr population et croissance

In [ ]:
df_pop = df_population.pivot_table(index=['Code zone', 'Zone'],columns='Année',values='Valeur',aggfunc='sum').reset_index()


In [ ]:
df_pop['Croissance sur 5 ans'] = ((df_pop[2017] - df_pop[2012]) / df_pop[2012])

In [ ]:
df_pop = df_pop.dropna(subset=[2012, 2017])

In [ ]:
df_pop = df_pop[['Code zone','Zone',2017,'Croissance sur 5 ans']]

In [ ]:
#df_pop = df_pop.rename(columns={2017:'Population (en Milliers)'}) (retiré)
#del df_pop[2017]

In [ ]:
df_pop

Année,Code zone,Zone,2017,Croissance sur 5 ans
0,1,Arménie,2944.791,0.020996
1,2,Afghanistan,36296.113,0.164779
2,3,Albanie,2884.169,-0.010270
3,4,Algérie,41389.189,0.107140
4,5,Samoa américaines,55.620,-0.000844
...,...,...,...,...
233,279,Curaçao,161.997,0.046377
234,280,Sint Maarten (partie néerlandaise),41.444,0.143567
235,281,Saint-Martin (partie française),36.560,-0.012132
236,282,Saint-Barthélemy,9.784,0.036550


In [ ]:
df_pop[2017] = df_pop[2017] * 1000
df_pop = df_pop.rename(columns={2017:'Population'})
df_pop.head()

Année,Code zone,Zone,Population,Croissance sur 5 ans
0,1,Arménie,2944791.0,0.020996
1,2,Afghanistan,36296113.0,0.164779
2,3,Albanie,2884169.0,-0.010270
3,4,Algérie,41389189.0,0.107140
4,5,Samoa américaines,55620.0,-0.000844


In [ ]:
df_pop.info()

<class 'pandas.core.frame.DataFrame'>
Index: 236 entries, 0 to 237
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Code zone             236 non-null    int64  
 1   Zone                  236 non-null    object 
 2   Population            236 non-null    float64
 3   Croissance sur 5 ans  236 non-null    float64
dtypes: float64(2), int64(1), object(1)
memory usage: 9.2+ KB


In [ ]:
#importation iso3
df_pop_iso3 = pd.merge(df_pop, pd.DataFrame(list(fao_to_iso_map.items()), columns=['Code zone', 'Code ISO3']), on='Code zone', how='left')

In [ ]:
df_pop_iso3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 236 entries, 0 to 235
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Code zone             236 non-null    int64  
 1   Zone                  236 non-null    object 
 2   Population            236 non-null    float64
 3   Croissance sur 5 ans  236 non-null    float64
 4   Code ISO3             228 non-null    object 
dtypes: float64(2), int64(1), object(2)
memory usage: 9.3+ KB


In [ ]:
df_pop_iso3_null = df_pop_iso3.loc[df_pop_iso3['Code ISO3'].isna()]
df_pop_iso3_null

,Code zone,Zone,Population,Croissance sur 5 ans,Code ISO3
134,151,Antilles néerlandaises (ex),275186.0,0.056993,NaN
165,187,"Sainte-Hélène, Ascension et Tristan da Cunha",6008.0,0.109716,NaN
223,259,Îles Anglo-Normandes,168665.0,0.040782,NaN
230,278,"Bonaire, Saint-Eustache et Saba",25401.0,0.112030,NaN
231,279,Curaçao,161997.0,0.046377,NaN
232,280,Sint Maarten (partie néerlandaise),41444.0,0.143567,NaN
233,281,Saint-Martin (partie française),36560.0,-0.012132,NaN
234,282,Saint-Barthélemy,9784.0,0.036550,NaN


Ok on peut les supprimer

In [ ]:
df_pop_iso3 = df_pop_iso3.dropna(subset=['Code ISO3'])

In [ ]:
df_pop_iso3.info()

<class 'pandas.core.frame.DataFrame'>
Index: 228 entries, 0 to 235
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Code zone             228 non-null    int64  
 1   Zone                  228 non-null    object 
 2   Population            228 non-null    float64
 3   Croissance sur 5 ans  228 non-null    float64
 4   Code ISO3             228 non-null    object 
dtypes: float64(2), int64(1), object(2)
memory usage: 10.7+ KB


Data frame PIB

In [ ]:
df_pib = df_pib[['Country Name', 'Country Code', '2017 [YR2017]']]

In [ ]:
df_pib = df_pib.dropna(subset=['Country Code'])

In [ ]:
df_pib

,Country Name,Country Code,2017 [YR2017]
0,Afghanistan,AFG,525.469770891619
1,Albania,ALB,5006.36012951959
2,Algeria,DZA,4554.66753957828
3,American Samoa,ASM,11863.6839452565
4,Andorra,AND,40672.9717424447
...,...,...,...
261,Sub-Saharan Africa,SSF,1546.91774199323
262,Sub-Saharan Africa (excluding high income),SSA,1641.39239957439
263,Sub-Saharan Africa (IDA & IBRD countries),TSS,1546.91774199323
264,Upper middle income,UMC,8064.20583627107


In [ ]:
from os import rename
df_pib = df_pib.rename(columns={'2017 [YR2017]':'PIB par habitant'})

In [ ]:
# Conversion en nombres
df_pib['PIB par habitant'] = pd.to_numeric(df_pib['PIB par habitant'], errors='coerce')

In [ ]:
df_pib

,Country Name,Country Code,PIB par habitant
0,Afghanistan,AFG,525.469771
1,Albania,ALB,5006.360130
2,Algeria,DZA,4554.667540
3,American Samoa,ASM,11863.683945
4,Andorra,AND,40672.971742
...,...,...,...
261,Sub-Saharan Africa,SSF,1546.917742
262,Sub-Saharan Africa (excluding high income),SSA,1641.392400
263,Sub-Saharan Africa (IDA & IBRD countries),TSS,1546.917742
264,Upper middle income,UMC,8064.205836


In [ ]:
import requests
def afficher_conversions():
    url = "https://api.frankfurter.app/latest?from=EUR&to=USD"
    try:
        response = requests.get(url)
        data = response.json()
        taux_eur_usd = data['rates']['USD']
        taux_usd_eur = 1 / taux_eur_usd
        print(f"📈 --- Taux de change ---")
        print(f"1 Euro (EUR)    = {taux_eur_usd:.4f} Dollars (USD)")
        print(f"1 Dollar (USD)  = {taux_usd_eur:.4f} Euros (EUR)")

    except Exception as e:
        print(f"❌ Erreur : {e}")

def obtenir_taux():
    url = "https://api.frankfurter.app/latest?from=EUR&to=USD"
    data = requests.get(url).json()
    t_eur_usd = data['rates']['USD']
    return 1 / t_eur_usd

taux_usd_eur = obtenir_taux()
afficher_conversions()

📈 --- Taux de change ---
1 Euro (EUR)    = 1.1694 Dollars (USD)
1 Dollar (USD)  = 0.8551 Euros (EUR)


In [ ]:
#taux_usd_eur = 0.93

In [ ]:
df_pib['PIB par habitant'] = df_pib['PIB par habitant'] * taux_usd_eur

In [ ]:
df_pib

,Country Name,Country Code,PIB par habitant
0,Afghanistan,AFG,449.349898
1,Albania,ALB,4281.135736
2,Algeria,DZA,3894.875611
3,American Samoa,ASM,10145.103425
4,Andorra,AND,34781.060153
...,...,...,...
261,Sub-Saharan Africa,SSF,1322.830291
262,Sub-Saharan Africa (excluding high income),SSA,1403.619292
263,Sub-Saharan Africa (IDA & IBRD countries),TSS,1322.830291
264,Upper middle income,UMC,6896.020041


### Stabilité politique & Croissance

Data frame pour la stabilité politique

In [ ]:
df_stabilites_politiques = df_stabilites_politiques[['Country Name', 'Country Code', '2017 [YR2017]']]

In [ ]:
df_stabilites_politiques = df_stabilites_politiques.rename(columns={'2017 [YR2017]':'Stabilité politique 2017'})

In [ ]:
# Conversion en nombres
df_stabilites_politiques['Stabilité politique 2017'] = pd.to_numeric(df_stabilites_politiques['Stabilité politique 2017'], errors='coerce')

In [ ]:
df_stabilites_politiques = df_stabilites_politiques.dropna(subset=['Country Code'])

In [ ]:
df_stabilites_politiques

,Country Name,Country Code,Stabilité politique 2017
0,Afghanistan,AFG,-2.794976
1,Albania,ALB,0.373771
2,Algeria,DZA,-0.919614
3,American Samoa,ASM,1.184324
4,Andorra,AND,1.392890
...,...,...,...
209,Virgin Islands (U.S.),VIR,0.981491
210,West Bank and Gaza,PSE,-1.650646
211,"Yemen, Rep.",YEM,-2.934317
212,Zambia,ZMB,0.142043


Data frame pour l'evolution de la stabilité politique

In [ ]:
df_stabilites_politiques_evolution

,Series Name,Series Code,Country Name,Country Code,2012 [YR2012],2013 [YR2013],2014 [YR2014],2015 [YR2015],2016 [YR2016],2017 [YR2017]
0,Political Stability and Absence of Violence/Te...,PV.EST,Afghanistan,AFG,-2.41856145858765,-2.51934909820557,-2.41106843948364,-2.56262516975403,-2.6621561050415,-2.7949755191803
1,Political Stability and Absence of Violence/Te...,PV.EST,Albania,ALB,-0.143631592392921,0.0919297859072685,0.485986232757568,0.341639041900635,0.337447881698608,0.373770743608475
2,Political Stability and Absence of Violence/Te...,PV.EST,Algeria,DZA,-1.32504332065582,-1.20237147808075,-1.19053518772125,-1.09078657627106,-1.09974193572998,-0.919614315032959
3,Political Stability and Absence of Violence/Te...,PV.EST,American Samoa,ASM,0.952990829944611,0.928985774517059,1.08068346977234,1.15167808532715,1.15855598449707,1.18432366847992
4,Political Stability and Absence of Violence/Te...,PV.EST,Andorra,AND,1.29035115242004,1.28392601013184,1.28659331798553,1.36598539352417,1.38275027275085,1.39288973808289
...,...,...,...,...,...,...,...,...,...,...
214,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
215,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
216,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
217,Data from database: Worldwide Governance Indic...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_spe = df_stabilites_politiques_evolution

In [ ]:
df_spe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 219 entries, 0 to 218
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Series Name    216 non-null    object
 1   Series Code    214 non-null    object
 2   Country Name   214 non-null    object
 3   Country Code   214 non-null    object
 4   2012 [YR2012]  214 non-null    object
 5   2013 [YR2013]  214 non-null    object
 6   2014 [YR2014]  214 non-null    object
 7   2015 [YR2015]  214 non-null    object
 8   2016 [YR2016]  214 non-null    object
 9   2017 [YR2017]  214 non-null    object
dtypes: object(10)
memory usage: 17.2+ KB


In [ ]:
# Conversion en nombres
df_spe['2017 [YR2017]'] = pd.to_numeric(df_spe['2017 [YR2017]'], errors='coerce')
df_spe['2012 [YR2012]'] = pd.to_numeric(df_spe['2012 [YR2012]'], errors='coerce')

In [ ]:
df_spe['SP sur 5 ans'] = ((df_spe['2017 [YR2017]'] - df_spe['2012 [YR2012]']) / df_spe['2012 [YR2012]'])

In [ ]:
df_spe = df_spe.dropna(subset=['Country Code'])

In [ ]:
df_spe[['Country Name', 'Country Code', 'SP sur 5 ans']]

,Country Name,Country Code,SP sur 5 ans
0,Afghanistan,AFG,0.155636
1,Albania,ALB,-3.602288
2,Algeria,DZA,-0.305974
3,American Samoa,ASM,0.242744
4,Andorra,AND,0.079466
...,...,...,...
209,Virgin Islands (U.S.),VIR,0.444269
210,West Bank and Gaza,PSE,-0.156125
211,"Yemen, Rep.",YEM,0.207209
212,Zambia,ZMB,-0.785096


### Taxes

Data frame pour les taxes

In [ ]:
df_taxes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 195 entries, 0 to 194
Data columns (total 21 columns):
 #   Column                                                                           Non-Null Count  Dtype  
---  ------                                                                           --------------  -----  
 0   Country                                                                          195 non-null    object 
 1   ISO3                                                                             194 non-null    object 
 2   Value exported in 2024 (USD thousand)                                            91 non-null     float64
 3   Trade balance 2024 (USD thousand)                                                93 non-null     float64
 4   Share in France's exports (%)                                                    91 non-null     float64
 5   Quantity exported in 2024                                                        91 non-null     float64
 6   Quantity u

In [ ]:
df_taxes = df_taxes[['Country','ISO3', 'Average tariff (estimated) faced by France (%)']]

In [ ]:
df_taxes = df_taxes.dropna()

In [ ]:
df_taxes

,Country,ISO3,Average tariff (estimated) faced by France (%)
1,Belgium,BEL,0.0
2,Germany,DEU,0.0
3,Spain,ESP,0.0
4,Hungary,HUN,0.0
5,Netherlands,NLD,0.0
...,...,...,...
188,Saint Pierre and Miquelon,SPM,0.0
189,Montserrat,MSR,0.7
190,Guinea-Bissau,GNB,6.9
191,"Libya, State of",LBY,0.0


### Distance France (vol d'oiseau)

Data frame distance depuis la France

In [ ]:
df_distance

,Country,ISO3,Capital City,Latitude,Longitude,Distance_from_France_km
0,Afghanistan,AFG,Kabul,34.5289,69.1725,5581.149309
1,Albania,ALB,Tiranë (Tirana),41.3275,19.8189,1600.850853
2,Algeria,DZA,El Djazaïr (Algiers),36.7525,3.0420,1347.074786
3,American Samoa,ASM,Pago Pago,-14.2781,-170.7025,16117.919387
4,Andorra,AND,Andorra la Vella,42.5078,1.5211,708.888083
...,...,...,...,...,...,...
229,Wallis and Futuna Islands,WLF,Matu-Utu,-13.2816,-176.1745,16057.009639
230,Western Sahara,ESH,El Aaiún,27.1532,-13.2014,2759.234627
231,Yemen,YEM,Sana'a',15.3531,44.2078,5313.102642
232,Zambia,ZMB,Lusaka,-15.4134,28.2771,7590.807787


In [ ]:
df_distance = df_distance[['Country','ISO3', 'Distance_from_France_km']]

In [ ]:
df_distance.head()

,Country,ISO3,Distance_from_France_km
0,Afghanistan,AFG,5581.149309
1,Albania,ALB,1600.850853
2,Algeria,DZA,1347.074786
3,American Samoa,ASM,16117.919387
4,Andorra,AND,708.888083


### Electricite

Data frame % de Population ayant de l'électricité

In [ ]:
df_electricite

,Series Name,Series Code,Country Name,Country Code,2017 [YR2017]
0,Access to electricity (% of population),EG.ELC.ACCS.ZS,Afghanistan,AFG,97.7
1,Access to electricity (% of population),EG.ELC.ACCS.ZS,Albania,ALB,99.9
2,Access to electricity (% of population),EG.ELC.ACCS.ZS,Algeria,DZA,99.5
3,Access to electricity (% of population),EG.ELC.ACCS.ZS,American Samoa,ASM,..
4,Access to electricity (% of population),EG.ELC.ACCS.ZS,Andorra,AND,100
...,...,...,...,...,...
266,NaN,NaN,NaN,NaN,NaN
267,NaN,NaN,NaN,NaN,NaN
268,NaN,NaN,NaN,NaN,NaN
269,Data from database: World Development Indicators,NaN,NaN,NaN,NaN


In [ ]:
df_electricite = df_electricite[['Country Name', 'Country Code', '2017 [YR2017]']]

In [ ]:
df_electricite = df_electricite.rename(columns={'2017 [YR2017]':'Pop ayant elec 2017'})

In [ ]:
df_electricite = df_electricite.dropna(subset=['Country Code'])

In [ ]:
df_electricite

,Country Name,Country Code,Pop ayant elec 2017
0,Afghanistan,AFG,97.7
1,Albania,ALB,99.9
2,Algeria,DZA,99.5
3,American Samoa,ASM,..
4,Andorra,AND,100
...,...,...,...
261,Sub-Saharan Africa,SSF,43.7628194566177
262,Sub-Saharan Africa (excluding high income),SSA,43.6966106335647
263,Sub-Saharan Africa (IDA & IBRD countries),TSS,43.7628194566177
264,Upper middle income,UMC,99.1608269162571


### Accord EU

Data frame des pays qui font parti de l'UE ou non

In [ ]:
df_accord_eu

,Country,ISO3,Accord_EU
0,Aruba,ABW,Non
1,Afghanistan,AFG,Non
2,Angola,AGO,Non
3,Anguilla,AIA,Non
4,Åland Islands,ALA,Non
...,...,...,...
244,Samoa,WSM,Non
245,Yemen,YEM,Non
246,South Africa,ZAF,Non
247,Zambia,ZMB,Non


In [ ]:
df_accord_eu.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 249 entries, 0 to 248
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Country    249 non-null    object
 1   ISO3       249 non-null    object
 2   Accord_EU  249 non-null    object
dtypes: object(3)
memory usage: 6.0+ KB


In [ ]:
#conversion
df_accord_eu['Accord-EU'] = df_accord_eu['Accord_EU'].map({'Oui': 1, 'Non': 0})

In [ ]:
df_accord_eu

,Country,ISO3,Accord_EU,Accord-EU
0,Aruba,ABW,Non,0
1,Afghanistan,AFG,Non,0
2,Angola,AGO,Non,0
3,Anguilla,AIA,Non,0
4,Åland Islands,ALA,Non,0
...,...,...,...,...
244,Samoa,WSM,Non,0
245,Yemen,YEM,Non,0
246,South Africa,ZAF,Non,0
247,Zambia,ZMB,Non,0


In [ ]:
#Suppression colonne 'Accord_EU'
del df_accord_eu['Accord_EU']

## Liaison

In [ ]:
df_pays = pd.read_csv('/content/drive/MyDrive/OC/ISO3.csv',sep=',')

In [ ]:
df_pays.head()

,Nom,Code_ISO3
0,Afghanistan,AFG
1,Afrique du Sud,ZAF
2,Albanie,ALB
3,Algérie,DZA
4,Allemagne,DEU


In [ ]:
duplicated_iso3_df_pays = df_pays[df_pays.duplicated(subset=['Code_ISO3'], keep=False)]
print("Code_ISO3 en doublons dans df_pays :")
print(duplicated_iso3_df_pays.sort_values(by='Code_ISO3'))

Code_ISO3 en doublons dans df_pays :
Empty DataFrame
Columns: [Nom, Code_ISO3]
Index: []


In [ ]:
df_pays.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Nom        194 non-null    object
 1   Code_ISO3  194 non-null    object
dtypes: object(2)
memory usage: 3.2+ KB


In [ ]:
#liaison avec df_disponibilite_interieur_proteines_iso3
df_merge = pd.merge(df_pays, df_disponibilite_interieur_proteines_iso3, left_on='Code_ISO3', right_on='Code ISO3', how='left')

In [ ]:
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 6 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Nom                                      194 non-null    object 
 1   Code_ISO3                                194 non-null    object 
 2   Code zone                                165 non-null    float64
 3   Zone                                     165 non-null    object 
 4   Disponibilité Volailles (Milliers de t)  165 non-null    float64
 5   Code ISO3                                165 non-null    object 
dtypes: float64(2), object(4)
memory usage: 9.2+ KB


In [ ]:
duplicated_rows = df_merge[df_merge.duplicated(subset=['Code_ISO3'], keep=False)]
print(duplicated_rows.sort_values(by='Code_ISO3'))

Empty DataFrame
Columns: [Nom, Code_ISO3, Code zone, Zone, Disponibilité Volailles (Milliers de t), Code ISO3]
Index: []


In [ ]:
#supprimer les dounblons
df_merge = df_merge.drop_duplicates(subset=['Code_ISO3'])

In [ ]:
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 6 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Nom                                      194 non-null    object 
 1   Code_ISO3                                194 non-null    object 
 2   Code zone                                165 non-null    float64
 3   Zone                                     165 non-null    object 
 4   Disponibilité Volailles (Milliers de t)  165 non-null    float64
 5   Code ISO3                                165 non-null    object 
dtypes: float64(2), object(4)
memory usage: 9.2+ KB


In [ ]:
#liaison avec df_importation_volailles_iso3
df_merge = pd.merge(df_merge, df_importation_volailles_iso3, left_on='Code_ISO3', right_on='Code ISO3', how='left')

In [ ]:
#supprimer les dounblons
df_merge = df_merge.drop_duplicates(subset=['Code_ISO3'])

In [ ]:
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 10 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Nom                                      194 non-null    object 
 1   Code_ISO3                                194 non-null    object 
 2   Code zone_x                              165 non-null    float64
 3   Zone_x                                   165 non-null    object 
 4   Disponibilité Volailles (Milliers de t)  165 non-null    float64
 5   Code ISO3_x                              165 non-null    object 
 6   Code zone_y                              165 non-null    float64
 7   Zone_y                                   165 non-null    object 
 8   Importation volailles (Milliers de t)    165 non-null    float64
 9   Code ISO3_y                              165 non-null    object 
dtypes: float64(4), object(6)
memory usage: 15.3+ KB


In [ ]:
# Selection colonne Nom Code_ISO3 Disponibilité protéines animales (Milliers de t) Disponibilité volailles (Milliers de t)
df_merge = df_merge[['Nom', 'Code_ISO3', 'Disponibilité Volailles (Milliers de t)', 'Importation volailles (Milliers de t)']]

In [ ]:
df_merge.head()

,Nom,Code_ISO3,Disponibilité Volailles (Milliers de t),Importation volailles (Milliers de t)
0,Afghanistan,AFG,57.0,29.0
1,Afrique du Sud,ZAF,2118.0,514.0
2,Albanie,ALB,47.0,38.0
3,Algérie,DZA,277.0,2.0
4,Allemagne,DEU,1739.0,842.0


In [ ]:
# merge avec df_pop_iso3
df_merge = pd.merge(df_merge, df_pop_iso3[['Code ISO3', 'Croissance sur 5 ans','Population']], left_on='Code_ISO3', right_on='Code ISO3', how='left')

In [ ]:
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 7 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Nom                                      194 non-null    object 
 1   Code_ISO3                                194 non-null    object 
 2   Disponibilité Volailles (Milliers de t)  165 non-null    float64
 3   Importation volailles (Milliers de t)    165 non-null    float64
 4   Code ISO3                                194 non-null    object 
 5   Croissance sur 5 ans                     194 non-null    float64
 6   Population                               194 non-null    float64
dtypes: float64(4), object(3)
memory usage: 10.7+ KB


In [ ]:
del df_merge['Code ISO3']

In [ ]:
df_merge = df_merge.drop_duplicates(subset=['Code_ISO3'])

In [ ]:
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 6 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Nom                                      194 non-null    object 
 1   Code_ISO3                                194 non-null    object 
 2   Disponibilité Volailles (Milliers de t)  165 non-null    float64
 3   Importation volailles (Milliers de t)    165 non-null    float64
 4   Croissance sur 5 ans                     194 non-null    float64
 5   Population                               194 non-null    float64
dtypes: float64(4), object(2)
memory usage: 9.2+ KB


In [ ]:
# merge avec df_pib
df_merge = pd.merge(df_merge, df_pib[['Country Code', 'PIB par habitant']], left_on='Code_ISO3', right_on='Country Code', how='left')

In [ ]:
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 8 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Nom                                      194 non-null    object 
 1   Code_ISO3                                194 non-null    object 
 2   Disponibilité Volailles (Milliers de t)  165 non-null    float64
 3   Importation volailles (Milliers de t)    165 non-null    float64
 4   Croissance sur 5 ans                     194 non-null    float64
 5   Population                               194 non-null    float64
 6   Country Code                             193 non-null    object 
 7   PIB par habitant                         190 non-null    float64
dtypes: float64(5), object(3)
memory usage: 12.3+ KB


In [ ]:
# merge avec df_stabilites_politiques
df_merge = pd.merge(df_merge, df_stabilites_politiques[['Country Code', 'Stabilité politique 2017']], left_on='Code_ISO3', right_on='Country Code', how='left')

In [ ]:
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 10 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Nom                                      194 non-null    object 
 1   Code_ISO3                                194 non-null    object 
 2   Disponibilité Volailles (Milliers de t)  165 non-null    float64
 3   Importation volailles (Milliers de t)    165 non-null    float64
 4   Croissance sur 5 ans                     194 non-null    float64
 5   Population                               194 non-null    float64
 6   Country Code_x                           193 non-null    object 
 7   PIB par habitant                         190 non-null    float64
 8   Country Code_y                           193 non-null    object 
 9   Stabilité politique 2017                 193 non-null    float64
dtypes: float64(6), object(4)
memory usage: 15.3+ KB


In [ ]:
# merge avec df_spe
df_merge = pd.merge(df_merge, df_spe[['Country Code', 'SP sur 5 ans']], left_on='Code_ISO3', right_on='Country Code', how='left')

In [ ]:
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 12 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Nom                                      194 non-null    object 
 1   Code_ISO3                                194 non-null    object 
 2   Disponibilité Volailles (Milliers de t)  165 non-null    float64
 3   Importation volailles (Milliers de t)    165 non-null    float64
 4   Croissance sur 5 ans                     194 non-null    float64
 5   Population                               194 non-null    float64
 6   Country Code_x                           193 non-null    object 
 7   PIB par habitant                         190 non-null    float64
 8   Country Code_y                           193 non-null    object 
 9   Stabilité politique 2017                 193 non-null    float64
 10  Country Code                             193 non-n

In [ ]:
# merge avec df_taxes
df_merge = pd.merge(df_merge, df_taxes[['ISO3', 'Average tariff (estimated) faced by France (%)']], left_on='Code_ISO3', right_on='ISO3', how='left')

In [ ]:
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 14 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Nom                                             194 non-null    object 
 1   Code_ISO3                                       194 non-null    object 
 2   Disponibilité Volailles (Milliers de t)         165 non-null    float64
 3   Importation volailles (Milliers de t)           165 non-null    float64
 4   Croissance sur 5 ans                            194 non-null    float64
 5   Population                                      194 non-null    float64
 6   Country Code_x                                  193 non-null    object 
 7   PIB par habitant                                190 non-null    float64
 8   Country Code_y                                  193 non-null    object 
 9   Stabilité politique 2017                   

In [ ]:
#merge avec df_distance
df_merge = pd.merge(df_merge, df_distance[['ISO3', 'Distance_from_France_km']], left_on='Code_ISO3', right_on='ISO3', how='left')

In [ ]:
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 16 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Nom                                             194 non-null    object 
 1   Code_ISO3                                       194 non-null    object 
 2   Disponibilité Volailles (Milliers de t)         165 non-null    float64
 3   Importation volailles (Milliers de t)           165 non-null    float64
 4   Croissance sur 5 ans                            194 non-null    float64
 5   Population                                      194 non-null    float64
 6   Country Code_x                                  193 non-null    object 
 7   PIB par habitant                                190 non-null    float64
 8   Country Code_y                                  193 non-null    object 
 9   Stabilité politique 2017                   

In [ ]:
del df_merge['Country Code_x']
del df_merge['Country Code_y']
del df_merge['ISO3_x']
del df_merge['ISO3_y']
del df_merge['Country Code']

In [ ]:
#merge avec df_electricite
df_merge = pd.merge(df_merge, df_electricite[['Country Code', 'Pop ayant elec 2017']], left_on='Code_ISO3', right_on='Country Code', how='left')

In [ ]:
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 13 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Nom                                             194 non-null    object 
 1   Code_ISO3                                       194 non-null    object 
 2   Disponibilité Volailles (Milliers de t)         165 non-null    float64
 3   Importation volailles (Milliers de t)           165 non-null    float64
 4   Croissance sur 5 ans                            194 non-null    float64
 5   Population                                      194 non-null    float64
 6   PIB par habitant                                190 non-null    float64
 7   Stabilité politique 2017                        193 non-null    float64
 8   SP sur 5 ans                                    193 non-null    float64
 9   Average tariff (estimated) faced by France 

In [ ]:
#merge avec df_accord_eu
df_merge = pd.merge(df_merge, df_accord_eu[['ISO3', 'Accord-EU']], left_on='Code_ISO3', right_on='ISO3', how='left')

In [ ]:
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 15 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Nom                                             194 non-null    object 
 1   Code_ISO3                                       194 non-null    object 
 2   Disponibilité Volailles (Milliers de t)         165 non-null    float64
 3   Importation volailles (Milliers de t)           165 non-null    float64
 4   Croissance sur 5 ans                            194 non-null    float64
 5   Population                                      194 non-null    float64
 6   PIB par habitant                                190 non-null    float64
 7   Stabilité politique 2017                        193 non-null    float64
 8   SP sur 5 ans                                    193 non-null    float64
 9   Average tariff (estimated) faced by France 

In [ ]:
del df_merge['ISO3']
del df_merge['Country Code']

In [ ]:
df_merge.head()

,Nom,Code_ISO3,Disponibilité Volailles (Milliers de t),Importation volailles (Milliers de t),Croissance sur 5 ans,Population,PIB par habitant,Stabilité politique 2017,SP sur 5 ans,Average tariff (estimated) faced by France (%),Distance_from_France_km,Pop ayant elec 2017,Accord-EU
0,Afghanistan,AFG,57.0,29.0,0.164779,36296113.0,449.349898,-2.794976,0.155636,5.0,5581.149309,97.7,0
1,Afrique du Sud,ZAF,2118.0,514.0,0.079063,57009756.0,5659.599010,-0.284804,10.218615,0.0,9341.821999,84.4,0
2,Albanie,ALB,47.0,38.0,-0.010270,2884169.0,4281.135736,0.373771,-3.602288,0.2,1600.850853,99.9,0
3,Algérie,DZA,277.0,2.0,0.107140,41389189.0,3894.875611,-0.919614,-0.305974,8.4,1347.074786,99.5,1
4,Allemagne,DEU,1739.0,842.0,0.020819,82658409.0,38954.963357,0.574381,-0.259457,0.0,877.998427,100,1


## SELECTION FINALE

In [ ]:
nan_countries_list = df_merge[df_merge.isnull().any(axis=1)]['Nom'].tolist()
print(nan_countries_list)

['Andorre', 'Australie', 'Bahreïn', 'Bangladesh', 'Barbade', 'Bhoutan', 'Brunei', 'Burundi', 'Comores', 'Congo (Kinshasa)', 'Corée du Nord', 'Cuba', 'Djibouti', 'Érythrée', 'France', 'Guinée équatoriale', 'Irak', 'Islande', 'Laos', 'Libye', 'Liechtenstein', 'Micronésie', 'Monaco', 'Nauru', 'Nigéria', 'Nouvelle-Zélande', 'Ouzbékistan', 'Palaos', 'Palestine', 'Papouasie-Nouvelle-Guinée', 'Qatar', 'Saint-Marin', 'Salomon', 'Seychelles', 'Singapour', 'Somalie', 'Soudan', 'Soudan du Sud', 'Syrie', 'Tonga', 'Turkménistan', 'Tuvalu', 'Vatican']


In [ ]:
#Enlever les NaN
df_merge_clean = df_merge.dropna()

In [ ]:
df_merge_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 151 entries, 0 to 193
Data columns (total 13 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Nom                                             151 non-null    object 
 1   Code_ISO3                                       151 non-null    object 
 2   Disponibilité Volailles (Milliers de t)         151 non-null    float64
 3   Importation volailles (Milliers de t)           151 non-null    float64
 4   Croissance sur 5 ans                            151 non-null    float64
 5   Population                                      151 non-null    float64
 6   PIB par habitant                                151 non-null    float64
 7   Stabilité politique 2017                        151 non-null    float64
 8   SP sur 5 ans                                    151 non-null    float64
 9   Average tariff (estimated) faced by France (%)  

In [ ]:
df_merge_clean['Disponibilité Volailles (Milliers de t)'] = df_merge_clean['Disponibilité Volailles (Milliers de t)'] * 1000
df_merge_clean['Importation volailles (Milliers de t)'] = df_merge_clean['Importation volailles (Milliers de t)'] * 1000

df_merge_clean = df_merge_clean.rename(columns={
    'Disponibilité Volailles (Milliers de t)': 'Disponibilité Volailles',
    'Importation volailles (Milliers de t)': 'Importation volailles'
})

df_merge_clean.head()

/tmp/ipykernel_33777/4293498520.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_merge_clean['Disponibilité Volailles (Milliers de t)'] = df_merge_clean['Disponibilité Volailles (Milliers de t)'] * 1000
/tmp/ipykernel_33777/4293498520.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_merge_clean['Importation volailles (Milliers de t)'] = df_merge_clean['Importation volailles (Milliers de t)'] * 1000


,Nom,Code_ISO3,Disponibilité Volailles,Importation volailles,Croissance sur 5 ans,Population,PIB par habitant,Stabilité politique 2017,SP sur 5 ans,Average tariff (estimated) faced by France (%),Distance_from_France_km,Pop ayant elec 2017,Accord-EU
0,Afghanistan,AFG,57000.0,29000.0,0.164779,36296113.0,449.349898,-2.794976,0.155636,5.0,5581.149309,97.7,0
1,Afrique du Sud,ZAF,2118000.0,514000.0,0.079063,57009756.0,5659.599010,-0.284804,10.218615,0.0,9341.821999,84.4,0
2,Albanie,ALB,47000.0,38000.0,-0.010270,2884169.0,4281.135736,0.373771,-3.602288,0.2,1600.850853,99.9,0
3,Algérie,DZA,277000.0,2000.0,0.107140,41389189.0,3894.875611,-0.919614,-0.305974,8.4,1347.074786,99.5,1
4,Allemagne,DEU,1739000.0,842000.0,0.020819,82658409.0,38954.963357,0.574381,-0.259457,0.0,877.998427,100,1


In [ ]:
df_merge_clean['Disponibilité Volailles (par habitant)'] = df_merge_clean['Disponibilité Volailles'] / df_merge_clean['Population']
df_merge_clean['Importation volailles (par habitant)'] = df_merge_clean['Importation volailles'] / df_merge_clean['Population']

df_merge_clean = df_merge_clean.drop(columns=['Population', 'Disponibilité Volailles', 'Importation volailles'])

df_merge_clean.head()

,Nom,Code_ISO3,Croissance sur 5 ans,PIB par habitant,Stabilité politique 2017,SP sur 5 ans,Average tariff (estimated) faced by France (%),Distance_from_France_km,Pop ayant elec 2017,Accord-EU,Disponibilité Volailles (par habitant),Importation volailles (par habitant)
0,Afghanistan,AFG,0.164779,449.349898,-2.794976,0.155636,5.0,5581.149309,97.7,0,0.001570,0.000799
1,Afrique du Sud,ZAF,0.079063,5659.599010,-0.284804,10.218615,0.0,9341.821999,84.4,0,0.037152,0.009016
2,Albanie,ALB,-0.010270,4281.135736,0.373771,-3.602288,0.2,1600.850853,99.9,0,0.016296,0.013175
3,Algérie,DZA,0.107140,3894.875611,-0.919614,-0.305974,8.4,1347.074786,99.5,1,0.006693,0.000048
4,Allemagne,DEU,0.020819,38954.963357,0.574381,-0.259457,0.0,877.998427,100,1,0.021038,0.010187


In [ ]:
df_merge_clean = df_merge_clean[df_merge_clean['Code_ISO3'] != 'PHL']
print(f"Le pays avec le code ISO3 'PHL' a été supprimé. Nouvelle taille du DataFrame: {len(df_merge_clean)}")

Le pays avec le code ISO3 'PHL' a été supprimé. Nouvelle taille du DataFrame: 150


Note : Les Philippines ont été exclues de ce résumé car elles constituaient un cas à part (Outlier) avec des taxes exceptionnellement élevées.

In [ ]:
df_merge_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 150 entries, 0 to 193
Data columns (total 12 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Nom                                             150 non-null    object 
 1   Code_ISO3                                       150 non-null    object 
 2   Croissance sur 5 ans                            150 non-null    float64
 3   PIB par habitant                                150 non-null    float64
 4   Stabilité politique 2017                        150 non-null    float64
 5   SP sur 5 ans                                    150 non-null    float64
 6   Average tariff (estimated) faced by France (%)  150 non-null    float64
 7   Distance_from_France_km                         150 non-null    float64
 8   Pop ayant elec 2017                             150 non-null    object 
 9   Accord-EU                                       

In [ ]:
#export du fichier
df_merge_clean.to_csv('df_OC-11_data.csv', index=False, encoding='utf-8')

In [ ]:
# telecharger
from google.colab import files
files.download('df_OC-11_data.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>